# HEF only — ft_v2 `best.onnx` → HEF in Hailo's own yolo26 layout

For a Google account **other than the one the model was trained on**. Drag
the laptop's folder `drone_training/drone_hef_v2zoo/` whole into **My Drive**, so
that **`MyDrive/drone_hef_v2zoo/`** holds:
- `best.onnx` — **ft_v2**. The notebook checks its md5 `a3dba8517af77bc0531cf3c8ad94ea11` and
  refuses any other model (ft_v3's has a different one)
- `hailo_dataflow_compiler-3.33.1-py3-none-linux_x86_64.whl`
- `calib_src.zip` — the 1024 calibration images (the same file as for ft_v2's current HEF)
- this notebook

Then `Runtime → Change runtime type → T4 GPU` and **Run all**. Two HEFs are
built one after the other into `MyDrive/drone_hef_v2zoo/hef_zoo/`:
1. **`zoo_fast`** — Hailo's layout (no forced context switch, 16-bit heads),
   the optimization of the current HEF. Its last build took ~37 min on a T4.
2. **`zoo`** — Hailo's script as is, + AdaRound. Its duration on Colab is
   **not known**, and it may run out of memory. If it fails, `zoo_fast` is
   already saved.

After a disconnect, Run all again: a HEF already there is skipped.

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, ('NO GPU on this runtime: Runtime -> Change runtime type -> T4 GPU. '
                           'If Colab offers Pro instead, this account is out of free GPU time.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, hashlib, os
INPUTS = '/content/drive/MyDrive/drone_hef_v2zoo'
assert os.path.isdir(INPUTS), f'{INPUTS} not found - drag the whole folder drone_hef_v2zoo into My Drive'
for pattern in ('best.onnx', 'hailo_dataflow_compiler-*.whl', 'calib_src.zip'):
    assert glob.glob(f'{INPUTS}/{pattern}'), f'put {pattern} in {INPUTS}'
onnxes = glob.glob(f'{INPUTS}/**/best.onnx', recursive=True)
assert len(onnxes) == 1, f'keep only ONE best.onnx (ft_v2) in {INPUTS}: {onnxes}'
md5 = hashlib.md5(open(onnxes[0], 'rb').read()).hexdigest()
assert md5 == 'a3dba8517af77bc0531cf3c8ad94ea11', f'best.onnx md5 {md5} - this is NOT ft_v2 (ft_v2 is a3dba8517af77bc0531cf3c8ad94ea11)'
print('inputs OK, best.onnx is ft_v2')

In [ ]:
%%writefile /content/build_calib.py
#!/usr/bin/env python3
"""
build_calib.py — the calibration set for the Hailo compile, taken from the
TRAIN split of the merged dataset and preprocessed exactly the way the
station hands a frame to the HEF.

    python build_calib.py <dataset_dir> <out_dir> [--n 1024] [--seed 0]
    python build_calib.py <dataset_dir> <out_dir> --pick-only   # list only
    python build_calib.py <dataset_dir> <out_dir> --pick calib_pick.txt

<dataset_dir>   the unzipped drone_dataset (images/train, labels/train)
<out_dir>/000000.npy ...   one image per file: float32 (640, 640, 3), RGB, 0..255
<out_dir>/calib_list.txt   which dataset image each file came from, and why
<out_dir>/calib_sheet.jpg  16 of them, for a look

WHY THESE CHOICES
  * TRAIN split only. The optimizer's quantization-aware fine-tune adjusts
    the quantized weights on these images; taking them from val would leak
    val into the model that the comparison is then run on.
  * Letterboxed to 640x640 with grey 114, RGB, values 0..255 — the same
    arithmetic as HailoInference._letterbox in TWO_CAMERAS_FIXED.py (the
    station's 640x480 frame lands between two 80-px grey bars). The chip
    divides by 255 itself (normalization in the model script), so values
    stay 0..255. The current HEF was calibrated on images STRETCHED to
    640x640 (dron_buster/hailo_recompile/build_calib.py) — a geometry the
    station never sends.
  * Stratified by drone size, so each of the three heads sees positives.
    Stride 8 decides the far, small targets; a set of mostly large drones
    would fit its class output's range to background alone. All train
    images without a drone (the station's own frames) are included, up to
    10 % of the set: they are the scene the chip actually looks at.
  * One file per image, not one 5 GB array: compile_hef.py stacks them into
    the in-memory array the current HEF was built from when RAM allows,
    and otherwise hands the Dataflow Compiler the directory itself (its
    "npy_dir" input: one image per .npy file).
"""
import argparse
import random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

SIZE = 640
PAD = 114
# Long side of the LARGEST drone in the frame, px after letterboxing to 640.
BUCKETS = [("<16", 0, 16, 0.25), ("16-32", 16, 32, 0.30),
           ("32-64", 32, 64, 0.25), (">=64", 64, 1e9, 0.20)]
MAX_BACKGROUND_SHARE = 0.10


def letterbox(img):
    """Copy of HailoInference._letterbox (TWO_CAMERAS_FIXED.py) — keep the
    two identical: calibration must see what the chip is fed."""
    sh, sw = img.shape[:2]
    r = min(SIZE / sh, SIZE / sw)
    nw, nh = int(round(sw * r)), int(round(sh * r))
    left = int(round((SIZE - nw) / 2.0 - 0.1))
    top = int(round((SIZE - nh) / 2.0 - 0.1))
    out = np.full((SIZE, SIZE, 3), PAD, dtype=np.uint8)
    if nw == sw and nh == sh:
        out[top:top + nh, left:left + nw] = img
    else:
        out[top:top + nh, left:left + nw] = cv2.resize(
            img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return out


def largest_drone_px(label_path, w, h):
    """Long side (px at 640) of the largest box, 0.0 for a background."""
    best = 0.0
    for line in label_path.read_text().splitlines():
        p = line.split()
        if len(p) >= 5:
            best = max(best, max(float(p[3]) * w, float(p[4]) * h))
    return best * SIZE / max(w, h)


def select(ds, n, seed):
    """[(image path, why)] — the calibration images, in the order they go
    into the set (the optimizer's statistics use the first 64)."""
    backgrounds, pools = [], {name: [] for name, *_ in BUCKETS}
    for img in sorted((ds / "images" / "train").glob("*.jpg")):
        lab = ds / "labels" / "train" / (img.stem + ".txt")
        if not lab.exists():
            continue
        with Image.open(img) as im:         # header only, no pixel decode
            w, h = im.size
        px = largest_drone_px(lab, w, h)
        if px == 0.0:
            backgrounds.append(img)
            continue
        for name, lo, hi, _share in BUCKETS:
            if lo <= px < hi:
                pools[name].append(img)
                break

    rng = random.Random(seed)
    n_bg = min(len(backgrounds), int(n * MAX_BACKGROUND_SHARE))
    picked = [(p, "background") for p in rng.sample(backgrounds, n_bg)]
    n_drone = n - n_bg
    for name, _lo, _hi, share in BUCKETS:
        take = min(int(round(n_drone * share)), len(pools[name]))
        rng.shuffle(pools[name])
        picked += [(p, name) for p in pools[name][:take]]
        pools[name] = pools[name][take:]
    # Rounding, or a bucket with too few images: fill from what is left.
    rest = [(p, name) for name, v in pools.items() for p in v]
    rng.shuffle(rest)
    picked += rest[:max(0, n - len(picked))]
    rng.shuffle(picked)
    return picked[:n]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dataset_dir")
    ap.add_argument("out_dir")
    ap.add_argument("--n", type=int, default=1024)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--pick-only", action="store_true",
                    help="write <out_dir>/calib_pick.txt (the selection, in "
                         "order) and stop — to ship only those images")
    ap.add_argument("--pick", help="a calib_pick.txt to follow instead of "
                                   "selecting: exactly those images, in order")
    args = ap.parse_args()
    ds, out = Path(args.dataset_dir), Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    if args.pick:
        picked = []
        for line in Path(args.pick).read_text().splitlines():
            why, name = line.split("\t")
            picked.append((ds / "images" / "train" / name, why))
    else:
        picked = select(ds, args.n, args.seed)
    if args.pick_only:
        (out / "calib_pick.txt").write_text(
            "".join(f"{why}\t{p.name}\n" for p, why in picked))
        print(f"{len(picked)} picked -> {out / 'calib_pick.txt'}")
        return
    for old in out.glob("*.npy"):
        old.unlink()

    counts = {}
    with open(out / "calib_list.txt", "w") as f:
        for i, (p, why) in enumerate(picked):
            bgr = cv2.imread(str(p))
            rgb = letterbox(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            np.save(out / f"{i:06d}.npy", rgb.astype(np.float32))
            f.write(f"{i:06d}.npy\t{why}\t{p.name}\n")
            counts[why] = counts.get(why, 0) + 1
    print(f"calibration set: {len(picked)} images -> {out}")
    print("  by largest drone (px at 640):",
          ", ".join(f"{k} {counts.get(k, 0)}"
                    for k in [b[0] for b in BUCKETS] + ["background"]))

    tiles = [cv2.resize(cv2.cvtColor(np.load(out / f"{i:06d}.npy")
                                     .astype(np.uint8), cv2.COLOR_RGB2BGR),
                        (320, 320)) for i in range(min(16, len(picked)))]
    while len(tiles) < 16:
        tiles.append(np.zeros((320, 320, 3), np.uint8))
    sheet = np.vstack([np.hstack(tiles[r * 4:r * 4 + 4]) for r in range(4)])
    cv2.imwrite(str(out / "calib_sheet.jpg"), sheet)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/compile_hef.py
#!/usr/bin/env python3
"""
compile_hef.py — ONNX -> HEF for the station's Hailo-8L with DFC 3.33.1,
the compiler the station's current HEF (bestty_yolo260508.hef) was built
with, so that a difference between HEFs comes from what was changed on
purpose, not from the compiler.

    <dfc venv>/bin/python compile_hef.py --onnx best.onnx --calib <dir> \\
        --out <dir> --recipe logits|sigmoid|zoo_fast|zoo
    <dfc venv>/bin/python compile_hef.py --from-har <...>_zoo_fast_quantized.har \\
        --out <dir> --recipe zoo_fast

--from-har  the compile alone, from the quantized HAR a run saves right
          after the optimization — for a session that died in the compile
          (Colab ran out of memory there after training, ft_v4 and ft_v5).
          Needs no GPU and no calibration set.

--calib   a directory of .npy files, one image each: float32 (640, 640, 3),
          RGB, 0..255, letterboxed like the station (build_calib.py)
--out     receives bestty_yolo26_<stamp>_<recipe>.hef, the quantized .har,
          the DFC logs and compile_summary_<recipe>.json
--recipe  logits   AS THE CURRENT HEF: the class outputs leave the chip as
                   logits and the station applies the sigmoid. Measured on
                   the Pi (25.09.2026): that HEF's class logits are
                   quantized in steps of ~0.69-0.79, so above 0.4 it can
                   only report 0.5 / 0.665 / 0.69 / 0.80 / 0.83 / 0.887 /
                   0.915, never more.
          sigmoid  the sigmoid moves ONTO the chip (conv64/80/94), so what
                   is quantized is 0..1 in steps of ~0.004. NOT USED: its
                   quantization-aware fine-tune diverges on this model —
                   28.09.2026, total distill loss 2.09 -> 6.51 -> 5.57 ->
                   5.34 over the 4 epochs, conv94's pinned at 1.0000,
                   while `logits` fell 1.17 -> 1.13 -> 1.04 -> 1.00; an
                   earlier compile this way (5.08.2026) logged an SNR of
                   -1.1 dB on the same conv94.

          zoo_fast HAILO'S OWN LAYOUT, our optimization. Hailo's model-zoo
                   script for yolo26n/yolo26s on Hailo-8L (hailo_model_zoo
                   v2.18 = DFC 3.33.0 / HailoRT 4.23,
                   cfg/alls/generic/yolo26n.alls) sets NO
                   context_switch_param and keeps the six head convs, their
                   outputs and dw1/dw6/dw7/dw8 in 16 bit. Measured on the
                   station's Pi 5 (hailortcli, batch 1, 02.10.2026): their
                   yolo26n HEF has 4 contexts and runs 47.7 FPS (PCIe Gen 2)
                   / 78.2 FPS (Gen 3), ours (`logits`) 6 contexts, 37.8 /
                   65.2 FPS. Which of the differences makes the 4 contexts is
                   NOT known. The optimization stays the default one (level 2,
                   as `logits`), so this recipe only changes the layout.
          zoo      Hailo's script as it is: on top of zoo_fast, optimization
                   level 4 with AdaRound (calibset 1024, batch 8). The DFC's
                   own docstring calls AdaRound "time consuming and requires
                   strong system" — on Colab's 12.7 GB it may run out of
                   memory; its duration there is not known.

COMMON TO ALL (as the current HEF)
  * model name `bestty_yolo26`: the HEF's outputs are then named
    bestty_yolo26/conv61 ... conv94, which HEAD_CONFIG in
    TWO_CAMERAS_FIXED.py expects (another name and the station falls back
    to shape-based auto-detection);
  * the graph cut at the six one2one head convolutions (box + class for
    strides 8/16/32);
  * normalization([0,0,0],[255,255,255]): the station feeds UINT8 0..255;
  * context_switch_param(mode=enabled) — logits/sigmoid only;
  * the default optimization — level 2 with a GPU and >= 1024 images
    (statistics on 64 images, 4 epochs of fine-tune on 1024) — all but zoo.
  The station's decoder detects either output kind by itself.
  Unlike the current HEF, the layer noise analysis (a report, not a step
  that changes the model) is switched off: it ran out of memory on Colab.

REFUSES rather than quietly producing a worse HEF when
  * an end node or one of the six output layers is not where it should be;
  * TensorFlow in this venv sees no GPU, or there are fewer than 1024
    calibration images: the optimizer would silently drop to level 0 (no
    quantization-aware fine-tune) and say so only in its log.
"""
import argparse
import json
import os
import re
import shutil
import sys
import time
from pathlib import Path

os.environ.setdefault("USER", "colab")        # the DFC reads it
# ⚠️ GPU selection, before the DFC is imported.
#  * Ultralytics' ONNX export calls select_device("cpu"), which sets
#    CUDA_VISIBLE_DEVICES="" in the notebook's process, and every command
#    started from the notebook afterwards inherits it: TensorFlow saw no GPU
#    on a T4 runtime (Colab, 25 and 28.09.2026). An empty or negative value
#    is therefore taken as "not chosen", not as "no GPU".
#  * The DFC's own selector (hailo_model_optimization/__init__.py) runs only
#    when the variable is unset, and rejects a GPU with > 5 % of its memory
#    in use — the notebook's PyTorch holds ~1.4 GB of the T4 — so it is
#    bypassed by naming the device.
if os.environ.get("CUDA_VISIBLE_DEVICES", "").strip() in ("", "-1"):
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "bestty_yolo26"
END_NODES = [
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv",
]
# (box layer, class layer, channels) per stride — HEAD_CONFIG's names.
HEADS = [("conv61", "conv64", 8), ("conv77", "conv80", 16),
         ("conv91", "conv94", 32)]
NORMALIZATION = (
    "norm_layer = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])\n")
# The layer noise analysis only REPORTS per-output SNR ("does not modify the
# structure of the model's graph", CheckerConfig in the 3.33.1 wheel). On
# Colab (12.7 GB) it was killed for memory (exit -9) right after the
# fine-tune had finished, twice (28.09.2026) — so it is off.
CHECKER_OFF = "model_optimization_config(checker_cfg, policy=disabled)\n"
MODEL_SCRIPT = (NORMALIZATION + "context_switch_param(mode=enabled)\n"
                + CHECKER_OFF)
# Hailo's 16-bit layers for yolo26n/s (cfg/alls/generic/yolo26n.alls, model
# zoo v2.18), names as in their script. check_layers() refuses to start if
# the parsed model does not have them.
ZOO_16BIT = (
    "quantization_param([dw1, dw6, dw7, dw8], precision_mode=a16_w16)\n"
    "quantization_param([conv61, conv77, conv91, conv64, conv80, conv94], "
    "precision_mode=a16_w16)\n"
    "quantization_param([output_layer1, output_layer2, output_layer3, "
    "output_layer4, output_layer5, output_layer6], precision_mode=a16_w16)\n"
)
RECIPES = {
    "logits": MODEL_SCRIPT,
    "sigmoid": MODEL_SCRIPT + "".join(
        f"change_output_activation({cls}, sigmoid)\n" for _, cls, _ in HEADS),
    "zoo_fast": NORMALIZATION + ZOO_16BIT + CHECKER_OFF,
    "zoo": (NORMALIZATION
            + "model_optimization_config(calibration, batch_size=8, "
              "calibset_size=1024)\n"
            + "model_optimization_flavor(optimization_level=4, "
              "compression_level=0)\n"
            + "post_quantization_optimization(adaround, policy=enabled, "
              "batch_size=8)\n"
            + ZOO_16BIT + CHECKER_OFF),
}
MIN_CALIB = 1024
IMG_BYTES = 640 * 640 * 3 * 4


def check_onnx(path):
    import onnx
    names = {n.name for n in onnx.load(path).graph.node}
    missing = [n for n in END_NODES if n not in names]
    if missing:
        heads = sorted(n for n in names if "one2one" in n and n.endswith("Conv"))
        sys.exit("end nodes not in the ONNX: %s\none2one convs present:\n  %s"
                 % (missing, "\n  ".join(heads)))


def check_heads(runner):
    """The six output layers must be HEAD_CONFIG's convs, box = 4 channels,
    class = 1 channel, at 80/40/20 cells. (Checked against the HN of the
    current model's HAR: output_layer1..6 <- conv61, 64, 77, 80, 91, 94.)"""
    layers = runner.get_hn_dict()["layers"]
    feeding = set()
    for layer in layers.values():
        if layer.get("type") == "output_layer":
            feeding.update(layer.get("input", []))
    problems = []
    for box, cls, stride in HEADS:
        cells = 640 // stride
        for short, ch in ((box, 4), (cls, 1)):
            full = f"{MODEL_NAME}/{short}"
            if full not in layers:
                problems.append(f"{full}: no such layer")
                continue
            if feeding and full not in feeding:
                problems.append(f"{full}: not an output")
            shape = (layers[full].get("output_shapes") or [[None]])[0]
            if list(shape[-3:]) != [cells, cells, ch]:
                problems.append(f"{full}: shape {shape}, expected "
                                f"[.., {cells}, {cells}, {ch}]")
    if problems:
        sys.exit("parsed model does not match HEAD_CONFIG:\n  "
                 + "\n  ".join(problems))
    print("heads OK: " + ", ".join(f"{b}/{c} (stride {s})"
                                   for b, c, s in HEADS))


def check_layers(runner, recipe):
    """Every layer a quantization_param names must exist in the parsed
    model — checked BEFORE the optimizer runs, so a name that differs from
    Hailo's yolo26n costs seconds, not an hour of Colab."""
    script = "".join(l for l in RECIPES[recipe].splitlines(True)
                     if l.startswith("quantization_param"))
    wanted = sorted(set(re.findall(r"\b(?:dw|conv|output_layer)\d+\b",
                                   script)))
    if not wanted:
        return
    layers = runner.get_hn_dict()["layers"]
    missing = [n for n in wanted if f"{MODEL_NAME}/{n}" not in layers]
    if missing:
        dws = sorted((n.split("/")[-1] for n in layers
                      if n.split("/")[-1].startswith("dw")),
                     key=lambda s: int(s[2:]) if s[2:].isdigit() else 0)
        sys.exit(f"recipe {recipe}: layers not in the parsed model: "
                 f"{missing}\n depthwise layers present: {dws}")
    print(f"16-bit layers OK: {', '.join(wanted)}")


def build_runner(onnx_path, recipe):
    from hailo_sdk_client import ClientRunner
    runner = ClientRunner(hw_arch="hailo8l")
    runner.translate_onnx_model(onnx_path, MODEL_NAME,
                                start_node_names=["images"],
                                end_node_names=END_NODES)
    check_heads(runner)
    check_layers(runner, recipe)
    runner.load_model_script(RECIPES[recipe])
    return runner


def calib_strategies(files):
    """Ways to hand the calibration set to runner.optimize, most proven
    first. Both are read by the DFC one image at a time through a
    generator (hailo_model_optimization/acceleras/utils/dataset_util.py in
    the 3.33.1 wheel), so the array costs its own size in RAM, not twice."""
    import numpy as np
    import psutil
    avail = psutil.virtual_memory().available
    # The array's 5 GB plus room for the optimizer and the compiler: with
    # only 4 GB of room the process was killed for memory on Colab.
    need = len(files) * IMG_BYTES + (10 << 30)

    def as_array():                 # what built the current HEF
        a = np.empty((len(files), 640, 640, 3), np.float32)
        for i, f in enumerate(files):
            a[i] = np.load(f)
        return a

    def as_dir():                   # DFC "npy_dir": one image per file,
        return str(files[0].parent)     # memory-mapped as it is read

    if avail >= need:
        return [("in-memory array", as_array), ("npy directory", as_dir)]
    print(f"RAM available {avail / 2**30:.1f} GB < {need / 2**30:.1f} GB "
          "for the in-memory array: the directory first")
    return [("npy directory", as_dir), ("in-memory array", as_array)]


def log_facts(text):
    """What the DFC's client log says about the optimization it ran."""
    lines = text.splitlines()
    return {
        "optimization_level": [l for l in lines
                               if "optimization level" in l.lower()][:5],
        "optimization_level_reduced": [l for l in lines
                                       if "Reducing optimization level" in l],
        "qft_lines": [l for l in lines if "for finetune" in l],
        "snr_db": dict(re.findall(r"(output_layer\d+) SNR:\s*(-?[\d.]+) dB",
                                  text)),
    }


def save_logs(out, recipe):
    for f in Path(".").glob("*.log"):
        shutil.copy(f, out / f"{recipe}_{f.name}")


def show_har(har):
    """A HAR is a tar archive: list it, and quote whatever its text members
    say about the optimization — the HAR is all that is left when the
    session that optimized it died before the compile finished."""
    import tarfile
    keys = re.compile(r"optimization_level|flavor|finetune|qft|Reducing|"
                      r"calibset|sdk_version|state", re.I)
    with tarfile.open(har) as t:
        for m in t.getmembers():
            print(f"  {m.name}  {m.size} B")
            if m.isfile() and m.size < (4 << 20) and not m.name.endswith(".npz"):
                text = t.extractfile(m).read().decode("utf-8", "replace")
                hits = [l.strip()[:200] for l in text.splitlines() if keys.search(l)]
                for l in hits[:12]:
                    print("     ", l)


def compile_from_har(args, out, dfc):
    """The compile alone, from a quantized HAR an earlier run saved: no GPU,
    no calibration set, no optimization — the weights are already the ones
    the optimizer produced. Only for recipes whose model script has no
    compile-time command (context_switch_param is one), so that compiling
    the HAR is the same as the compile that run would have done."""
    har = Path(args.from_har).resolve()
    if "context_switch_param" in RECIPES[args.recipe]:
        sys.exit(f"--from-har: recipe {args.recipe} has a compile-time command "
                 "that a HAR may not carry; optimize again instead")
    if not har.name.endswith(f"_{args.recipe}_quantized.har"):
        sys.exit(f"{har.name} is not a {args.recipe} quantized HAR")
    print("HAR", har)
    show_har(har)
    from hailo_sdk_client import ClientRunner
    runner = ClientRunner(har=str(har))
    check_heads(runner)
    t0 = time.time()
    hef = out / (har.name[:-len("_quantized.har")] + ".hef")
    hef.write_bytes(runner.compile())
    # The optimization's own log, if the run that wrote the HAR saved it.
    opt_log = out / f"{args.recipe}_hailo_sdk.client.log"
    facts = (log_facts(opt_log.read_text(errors="replace")) if opt_log.exists()
             else {"optimization_level": "NOT KNOWN: the log of the run that "
                   "optimized this HAR was not saved"})
    summary = {"hef": hef.name, "recipe": args.recipe, "dfc": dfc,
               "compiled_from_har": har.name,
               "model_script": RECIPES[args.recipe], **facts,
               "compile_min": round((time.time() - t0) / 60, 1)}
    (out / f"compile_summary_{args.recipe}.json").write_text(
        json.dumps(summary, indent=1))
    for f in Path(".").glob("*.log"):
        shutil.copy(f, out / f"{args.recipe}_compile_{f.name}")
    print(json.dumps(summary, indent=1))
    print(f"\nSUCCESS: {hef}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--onnx")
    ap.add_argument("--calib")
    ap.add_argument("--out", required=True)
    ap.add_argument("--recipe", choices=sorted(RECIPES), required=True)
    ap.add_argument("--work", help="scratch folder for the DFC (default: "
                    "/content/hef_work_<recipe> on Colab, else under --out)")
    ap.add_argument("--from-har", help="compile this <...>_<recipe>_quantized"
                    ".har (saved by a run whose compile did not finish) "
                    "instead of optimizing again; needs no GPU")
    args = ap.parse_args()
    if not args.from_har and not (args.onnx and args.calib):
        ap.error("--onnx and --calib are required unless --from-har")
    out = Path(args.out).resolve()
    out.mkdir(parents=True, exist_ok=True)
    work = Path(args.work) if args.work else (
        (Path("/content") if Path("/content").exists() else out)
        / f"hef_work_{args.recipe}")
    shutil.rmtree(work, ignore_errors=True)   # its logs must be this run's
    work.mkdir(parents=True)
    os.chdir(work)                      # the DFC writes its logs to the cwd

    import hailo_sdk_client
    import tensorflow as tf
    dfc = getattr(hailo_sdk_client, "__version__", "?")
    print("DFC", dfc, "| TensorFlow", tf.__version__)
    if args.from_har:
        compile_from_har(args, out, dfc)
        return
    onnx_path = str(Path(args.onnx).resolve())
    if not tf.config.list_physical_devices("GPU"):
        sys.exit("TensorFlow in this venv sees no GPU: the optimizer would "
                 "drop to level 0 (no quantization-aware fine-tune). Fix the "
                 "CUDA install first.")
    files = sorted(Path(args.calib).glob("*.npy"))
    if len(files) < MIN_CALIB:
        sys.exit(f"{len(files)} calibration images < {MIN_CALIB}: the "
                 "optimizer would drop to a lower level")
    check_onnx(onnx_path)

    t0 = time.time()
    runner, used, errors = None, None, []
    for label, make in calib_strategies(files):
        runner = build_runner(onnx_path, args.recipe)
        try:
            print(f"optimize: calibration as {label}", flush=True)
            runner.optimize(make())
            used = label
            break
        except Exception as exc:          # try the next way of feeding it
            errors.append(f"{label}: {type(exc).__name__}: {exc}")
            print("  failed —", errors[-1][:500], flush=True)
    if used is None:
        sys.exit("optimize failed every way:\n  " + "\n  ".join(errors))
    t_opt = time.time() - t0

    stamp = time.strftime("%m%d_%H%M")
    base = f"{MODEL_NAME}_{stamp}_{args.recipe}"
    runner.save_har(str(out / f"{base}_quantized.har"))
    # The optimization log goes next to the HAR NOW: a session that dies in
    # the compile (ft_v4 and ft_v5 did, out of memory) leaves the HAR, and
    # without this log nothing says at which level it was optimized.
    save_logs(out, args.recipe)
    hef = out / f"{base}.hef"
    hef.write_bytes(runner.compile())
    t_all = time.time() - t0

    log = Path("hailo_sdk.client.log")
    text = log.read_text(errors="replace") if log.exists() else ""
    summary = {
        "hef": hef.name, "recipe": args.recipe,
        "model_script": RECIPES[args.recipe], "dfc": dfc, "onnx": onnx_path,
        "calibration_images": len(files), "calibration_as": used,
        "failed_ways": errors, **log_facts(text),
        "optimize_min": round(t_opt / 60, 1),
        "total_min": round(t_all / 60, 1),
    }
    (out / f"compile_summary_{args.recipe}.json").write_text(
        json.dumps(summary, indent=1))
    save_logs(out, args.recipe)
    print(json.dumps(summary, indent=1))
    if summary["optimization_level_reduced"]:
        print("\n⚠️  THE OPTIMIZER REDUCED ITS LEVEL — this HEF is not built "
              "the way the current one was. Do not deploy it.")
    print(f"\nSUCCESS: {hef}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/hef_only.py
#!/usr/bin/env python3
"""
hef_only.py — just the HEF part of train_colab.ipynb (calibration set, the
DFC venv, both compiles) from three files, without the dataset or the
training run. For a runtime that cannot see the training account's Drive:
another Google account's Colab (hef_colab.ipynb), or a Kaggle kernel.

    python hef_only.py --inputs DIR --out DIR [--work DIR] [--recipes logits]

--inputs  searched recursively for
            hailo_dataflow_compiler-*.whl   the DFC 3.33.1 wheel
            best.onnx                       runs/ft_v2/weights on the Drive
                                            the model was trained on
            calib_src.zip                   the 1024 calibration images picked
                                            by `build_calib.py --pick-only`,
                                            with labels and calib_pick.txt
                                            (or that zip already unpacked)
          build_calib.py and compile_hef.py are taken from there too, or else
          from this script's own folder
--out     bestty_yolo26_<stamp>_<recipe>.hef, the quantized .har,
          compile_summary_*.json, the DFC logs. A recipe whose HEF is already
          there is skipped, so a rerun after a disconnect carries on.
--work    scratch: the unpacked images, the 5 GB calibration set, the venv
--recipes compile_hef.py recipes to build, in this order; only `logits` by
          default — the `sigmoid` recipe's fine-tune diverges. `zoo_fast` /
          `zoo` are Hailo's own layout for yolo26 (see compile_hef.py): put
          zoo_fast FIRST, so its HEF is on the Drive even if zoo's AdaRound
          runs out of memory or time
--onnx-md5 refuse to start unless best.onnx has this md5 (the Drive folder
          has held two different models' best.onnx)
"""
import argparse
import glob
import hashlib
import os
import subprocess
import sys
import zipfile
from pathlib import Path

HERE = Path(__file__).resolve().parent


def find(root, pattern, fallback=None):
    hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    if hits:
        return hits[0]
    if fallback and Path(fallback).exists():
        return str(fallback)
    sys.exit(f"not found under {root}: {pattern}")


def sh(cmd):
    print("+", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--inputs", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--work", default="/tmp/hef_only")
    ap.add_argument("--recipes", nargs="+", default=["logits"],
                    choices=["logits", "sigmoid", "zoo_fast", "zoo"])
    ap.add_argument("--onnx-md5")
    args = ap.parse_args()
    inp, out, work = Path(args.inputs), Path(args.out), Path(args.work)
    out.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)

    gpu = subprocess.run(
        "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
        shell=True, capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip() or gpu.stderr.strip(), flush=True)
    if gpu.returncode != 0:
        sys.exit("no GPU on this runtime: the compile would lose the "
                 "quantization-aware fine-tune the current HEF has")

    todo = [r for r in args.recipes if not glob.glob(f"{out}/*_{r}.hef")]
    print("HEF still to build:", todo or "none", flush=True)
    if not todo:
        return

    wheel = find(inp, "hailo_dataflow_compiler-*.whl")
    onnxes = sorted(glob.glob(f"{inp}/**/best.onnx", recursive=True))
    if len(onnxes) > 1:
        sys.exit(f"more than one best.onnx under {inp} - keep only the "
                 f"model to compile: {onnxes}")
    onnx = find(inp, "best.onnx")
    if args.onnx_md5:
        md5 = hashlib.md5(Path(onnx).read_bytes()).hexdigest()
        if md5 != args.onnx_md5:
            sys.exit(f"{onnx} has md5 {md5}, not {args.onnx_md5}: "
                     "this is not the model this run is meant to compile")
        print("best.onnx md5 OK:", md5, flush=True)
    build_calib = find(inp, "build_calib.py", HERE / "build_calib.py")
    compile_hef = find(inp, "compile_hef.py", HERE / "compile_hef.py")

    picks = glob.glob(f"{inp}/**/calib_pick.txt", recursive=True)
    if picks:                                   # the zip came unpacked
        src = Path(picks[0]).parent
    else:
        src = work / "calib_src"
        if not (src / "calib_pick.txt").exists():
            with zipfile.ZipFile(find(inp, "calib_src.zip")) as z:
                z.extractall(src)
    calib = work / "calib"
    if len(list(calib.glob("*.npy"))) < 1024:
        sh(f'"{sys.executable}" "{build_calib}" "{src}" "{calib}" '
           f'--pick "{src}/calib_pick.txt"')

    # The DFC in its own Python 3.10 venv: it pins TensorFlow 2.18 and a
    # matplotlib (3.5.2) whose wheels stop at Python 3.10.
    vpy = work / "hailo_venv" / "bin" / "python"
    uv = f'"{sys.executable}" -m uv'
    sh(f'"{sys.executable}" -m pip -q install uv')
    sh(f'{uv} venv -q --allow-existing --python 3.10 "{work}/hailo_venv"')
    sh(f'{uv} pip install -q --python "{vpy}" "{wheel}" psutil')
    tf = subprocess.check_output(
        [str(vpy), "-c", "import tensorflow as t; print(t.__version__)"],
        stderr=subprocess.DEVNULL, text=True).strip()
    print("DFC pins TensorFlow", tf, "- adding its CUDA libraries", flush=True)
    sh(f'{uv} pip install -q --python "{vpy}" "tensorflow[and-cuda]=={tf}"')

    # Named explicitly: an inherited empty CUDA_VISIBLE_DEVICES hides the GPU.
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
    failed = []
    for recipe in todo:
        r = subprocess.run([str(vpy), compile_hef, "--recipe", recipe,
                            "--onnx", onnx, "--calib", str(calib),
                            "--out", str(out),
                            "--work", str(work / f"hef_work_{recipe}")],
                           env=env)
        print(f"== {recipe}: exit {r.returncode}", flush=True)
        if r.returncode:
            failed.append(recipe)
    sh(f'ls -la "{out}"')
    if failed:
        sys.exit(f"failed: {failed}")


if __name__ == "__main__":
    main()

In [ ]:
!python /content/hef_only.py --inputs "{INPUTS}" --out "{INPUTS}/hef_zoo" --work /content/hef_tmp --recipes zoo_fast zoo --onnx-md5 a3dba8517af77bc0531cf3c8ad94ea11

## Next
Download everything from `MyDrive/drone_hef_v2zoo/hef_zoo/` (the `.hef` files and
`compile_summary_*.json`) → `D:/d/DocumentsFromC/GitHub/drone_training/new_model_zoo/`.
Before anything goes on the station, each HEF is measured on the Pi:
contexts and FPS (`hailortcli`), and detection quality against ft_v2's
current HEF (`compare_hef.py`).